In [5]:
import pandas as pd
import re
from rdflib import Graph, Namespace, RDF, RDFS, XSD, Literal, URIRef


In [6]:
# Load new prepared class files
listing_df = pd.read_csv("data/class_Listing.csv")
host_df = pd.read_csv("data/class_Host.csv")
borough_df = pd.read_csv("data/class_Borough.csv")

# Load pressure files
airbnb_pressure = pd.read_csv("data/airbnb_pressure_by_borough.csv")
housing_pressure = pd.read_csv("data/housing_pressure_by_borough.csv")
combined_pressure = pd.read_csv("data/combined_pressure_by_borough.csv")

In [8]:
# Create graph
g = Graph()
EX = Namespace("http://example.org/london-airbnb/")

g.bind("ex", EX)
g.bind("rdf", RDF)
g.bind("rdfs", RDFS)
g.bind("xsd", XSD)

def removeSpace(value):
    return str(value).replace(" ", "_").replace("/", "_")

# Classes
classes = [
    "Borough",
    "Listing",
    "Host",
    "RoomType",
    "HousingIndicator",
    "PressureIndicator"
]

for cls in classes:
    g.add((EX[cls], RDF.type, RDFS.Class))

# Object properties
object_properties = {
    "isLocatedIn": ("Listing", "Borough"),
    "hasRoomType": ("Listing", "RoomType"),
    "hasListing": ("Host", "Listing"),
    "hasHousingIndicator": ("Borough", "HousingIndicator"),
    "hasPressureIndicator": ("Borough", "PressureIndicator"),
}

for prop, (domain, range_) in object_properties.items():
    g.add((EX[prop], RDF.type, RDF.Property))
    g.add((EX[prop], RDFS.domain, EX[domain]))
    g.add((EX[prop], RDFS.range, EX[range_]))

# Datatype properties
datatype_properties = {
    # Listing
    "listingID": ("Listing", XSD.integer),
    "priceNight": ("Listing", XSD.float),
    "availability": ("Listing", XSD.integer),
    "reviewsMonth": ("Listing", XSD.float),

    # Host
    "hostID": ("Host", XSD.integer),
    "hostListingCount": ("Host", XSD.integer),

    # Borough
    "boroughName": ("Borough", XSD.string),
    "populationEstimate": ("Borough", XSD.integer),
    "householdEstimate": ("Borough", XSD.integer),
    "populationDensity": ("Borough", XSD.float),
    "medianHousePrice": ("Borough", XSD.float),
    "newHomes": ("Borough", XSD.float),
    "medianIncome": ("Borough", XSD.float),
    "ownedRatio": ("Borough", XSD.float),
    "rentedAssociationRatio": ("Borough", XSD.float),
    "transportAccessibility": ("Borough", XSD.float),
    "rentedPrivateRatio": ("Borough", XSD.float),

    # RoomType
    "roomTypeName": ("RoomType", XSD.string),

    # Indicators
    "airbnbPressureScore": ("PressureIndicator", XSD.float),
    "airbnbPressureLevel": ("PressureIndicator", XSD.string),
    "housingPressureScore": ("HousingIndicator", XSD.float),
    "housingPressureLevel": ("HousingIndicator", XSD.string),
}

for prop, (domain, range_) in datatype_properties.items():
    g.add((EX[prop], RDF.type, RDF.Property))
    g.add((EX[prop], RDFS.domain, EX[domain]))
    g.add((EX[prop], RDFS.range, range_))

# Column mappings for new class files
listing_literal_mapping = {
    "listingID": (EX.listingID, XSD.integer),
    "priceNight": (EX.priceNight, XSD.float),
    "availability": (EX.availability, XSD.integer),
    "reviewsMonth": (EX.reviewsMonth, XSD.float),
}

host_literal_mapping = {
    "hostID": (EX.hostID, XSD.integer),
    "hostListingCount": (EX.hostListingCount, XSD.integer),
}

borough_literal_mapping = {
    "populationEstimate": (EX.populationEstimate, XSD.integer),
    "householdEstimate": (EX.householdEstimate, XSD.integer),
    "populationDensity": (EX.populationDensity, XSD.float),
    "medianHousePrice": (EX.medianHousePrice, XSD.float),
    "newHomes": (EX.newHomes, XSD.float),
    "medianIncome": (EX.medianIncome, XSD.float),
    "ownedRatio": (EX.ownedRatio, XSD.float),
    "rentedAssociationRatio": (EX.rentedAssociationRatio, XSD.float),
    "transportScore": (EX.transportAccessibility, XSD.float),
    "rentedPrivateRatio": (EX.rentedPrivateRatio, XSD.float),
}

# Convert class_Listing
for _, row in listing_df.iterrows():

    listing = EX[f"listing/{row['listingID']}"]
    borough = EX[f"borough/{removeSpace(row['borough'])}"]
    room_type = EX[f"roomType/{removeSpace(row['roomType'])}"]

    g.add((listing, RDF.type, EX.Listing))
    g.add((borough, RDF.type, EX.Borough))
    g.add((room_type, RDF.type, EX.RoomType))

    g.add((listing, EX.isLocatedIn, borough))
    g.add((listing, EX.hasRoomType, room_type))

    g.add((borough, EX.boroughName, Literal(row["borough"], datatype=XSD.string)))
    g.add((room_type, EX.roomTypeName, Literal(row["roomType"], datatype=XSD.string)))

    for csv_col, (rdf_prop, datatype) in listing_literal_mapping.items():
        if csv_col in listing_df.columns and pd.notna(row[csv_col]):
            g.add((listing, rdf_prop, Literal(row[csv_col], datatype=datatype)))

# Convert class_Host
for _, row in host_df.iterrows():

    host = EX[f"host/{row['hostID']}"]
    listing = EX[f"listing/{row['listingID']}"]

    g.add((host, RDF.type, EX.Host))
    g.add((listing, RDF.type, EX.Listing))
    g.add((host, EX.hasListing, listing))

    for csv_col, (rdf_prop, datatype) in host_literal_mapping.items():
        if csv_col in host_df.columns and pd.notna(row[csv_col]):
            g.add((host, rdf_prop, Literal(row[csv_col], datatype=datatype)))

# Convert class_Borough
for _, row in borough_df.iterrows():

    borough = EX[f"borough/{removeSpace(row['borough'])}"]

    g.add((borough, RDF.type, EX.Borough))
    g.add((borough, EX.boroughName, Literal(row["borough"], datatype=XSD.string)))

    for csv_col, (rdf_prop, datatype) in borough_literal_mapping.items():
        if csv_col in borough_df.columns and pd.notna(row[csv_col]):
            g.add((borough, rdf_prop, Literal(row[csv_col], datatype=datatype)))

# Add pressure indicators from combined pressure file
for _, row in combined_pressure.iterrows():

    borough = EX[f"borough/{removeSpace(row['borough'])}"]
    pressure_indicator = EX[f"pressureIndicator/{removeSpace(row['borough'])}"]
    housing_indicator = EX[f"housingIndicator/{removeSpace(row['borough'])}"]

    g.add((borough, RDF.type, EX.Borough))

    g.add((pressure_indicator, RDF.type, EX.PressureIndicator))
    g.add((housing_indicator, RDF.type, EX.HousingIndicator))

    g.add((borough, EX.hasPressureIndicator, pressure_indicator))
    g.add((borough, EX.hasHousingIndicator, housing_indicator))

    if "airbnb_pressure_score" in combined_pressure.columns:
        g.add((pressure_indicator, EX.airbnbPressureScore,
               Literal(row["airbnb_pressure_score"], datatype=XSD.float)))

    if "airbnb_pressure_level" in combined_pressure.columns:
        g.add((pressure_indicator, EX.airbnbPressureLevel,
               Literal(row["airbnb_pressure_level"], datatype=XSD.string)))

    if "housing_pressure_score" in combined_pressure.columns:
        g.add((housing_indicator, EX.housingPressureScore,
               Literal(row["housing_pressure_score"], datatype=XSD.float)))

    if "housing_pressure_level" in combined_pressure.columns:
        g.add((housing_indicator, EX.housingPressureLevel,
               Literal(row["housing_pressure_level"], datatype=XSD.string)))

# Save RDF file
g.serialize(destination="london_airbnb_kg.ttl", format="turtle")

print("RDF graph created successfully.")
print("Total triples:", len(g))
print("Saved file:")
print("london_airbnb_kg.ttl")

# SPARQL test
query = """
PREFIX ex: <http://example.org/london-airbnb/>

SELECT ?borough ?indicator
WHERE {
    ?borough ex:hasPressureIndicator ?indicator .
}
LIMIT 10
"""

for result in g.query(query):
    print(result)

RDF graph created successfully.
Total triples: 831472
Saved file:
london_airbnb_kg.ttl
(rdflib.term.URIRef('http://example.org/london-airbnb/borough/Barking_and_Dagenham'), rdflib.term.URIRef('http://example.org/london-airbnb/pressureIndicator/Barking_and_Dagenham'))
(rdflib.term.URIRef('http://example.org/london-airbnb/borough/Barnet'), rdflib.term.URIRef('http://example.org/london-airbnb/pressureIndicator/Barnet'))
(rdflib.term.URIRef('http://example.org/london-airbnb/borough/Bexley'), rdflib.term.URIRef('http://example.org/london-airbnb/pressureIndicator/Bexley'))
(rdflib.term.URIRef('http://example.org/london-airbnb/borough/Brent'), rdflib.term.URIRef('http://example.org/london-airbnb/pressureIndicator/Brent'))
(rdflib.term.URIRef('http://example.org/london-airbnb/borough/Bromley'), rdflib.term.URIRef('http://example.org/london-airbnb/pressureIndicator/Bromley'))
(rdflib.term.URIRef('http://example.org/london-airbnb/borough/Camden'), rdflib.term.URIRef('http://example.org/london-ai